# Local LLMs with HuggingFace Transformers

**Goal:** Understand how to load and run Large Language Models locally
using the HuggingFace `transformers` library.

**Why run LLMs locally?**
- **Privacy**: data never leaves your machine
- **Cost**: no API fees for inference
- **Latency**: no network round-trips
- **Offline**: works without internet after model download

We will use **SmolLM2-360M-Instruct**, a tiny but capable
instruction-tuned language model from HuggingFace. It has only 360M
parameters, making it practical to run on any laptop CPU.

**Sources:**
- [HuggingFace LLM Tutorial](https://huggingface.co/docs/transformers/en/llm_tutorial)
- [HuggingFace smol-course](https://github.com/huggingface/smol-course)

In [ ]:
%pip install transformers torch -q

## 1. Tokenization

Before a language model can process text, the text must be converted
into a sequence of **token IDs** — integers that index into the
model's vocabulary.

The **tokenizer** handles this conversion. Modern LLMs use
**subword tokenization** (e.g., BPE — Byte Pair Encoding), which
splits text into frequent subword units. This means:
- Common words get a single token (e.g., `the` -> 1 token)
- Rare words are split into pieces (e.g., `tokenization` -> multiple tokens)
- The model can handle any text, including words it hasn't seen before

In [ ]:
from transformers import AutoTokenizer

model_name = "HuggingFaceTB/SmolLM2-360M-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)

### Encode and Decode

The tokenizer provides `encode()` to convert text to token IDs,
and `decode()` to convert back. Let's verify the round-trip:

In [ ]:
text = "Hello, how are you doing today?"

token_ids = tokenizer.encode(text)
print(f"Text:      {text!r}")
print(f"Token IDs: {token_ids}")
print(f"Decoded:   {tokenizer.decode(token_ids)!r}")

### Vocabulary size and special tokens

In [ ]:
print(f"Vocabulary size: {tokenizer.vocab_size:,}")
print(f"Special tokens:  {tokenizer.all_special_tokens}")
print(f"EOS token:       {tokenizer.eos_token!r} (id={tokenizer.eos_token_id})")
print(f"BOS token:       {tokenizer.bos_token!r} (id={tokenizer.bos_token_id})")

### Subword tokenization in action

Let's see how the tokenizer splits various strings into subwords.
Notice how common words stay intact while rare or compound words
get broken into pieces:

In [ ]:
examples = [
    "The cat sat on the mat.",
    "Tokenization is fascinating!",
    "Supercalifragilisticexpialidocious",
    "The transformer architecture uses self-attention.",
    "def fibonacci(n): return n if n < 2 else fibonacci(n-1) + fibonacci(n-2)",
    "GPT-4 and LLaMA are large language models.",
]

for text in examples:
    tokens = tokenizer.tokenize(text)
    print(f"\n{text!r}")
    print(f"  {len(tokens)} tokens: {tokens}")

## 2. Loading the Model

We use `AutoModelForCausalLM` from the `transformers` library to
load the model weights. This downloads the full-precision model
(~720 MB in float16) from HuggingFace Hub and caches it locally.

We load in **float16** to halve memory usage compared to float32,
with negligible quality loss.

In [ ]:
import torch
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    model_name, dtype=torch.float16
).to("cpu")
model.eval()
print(f"Model loaded: {model_name}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"dtype: {next(model.parameters()).dtype}")

### Model memory footprint

Let's estimate the model's memory usage based on its parameter count
and data type:

In [ ]:
n_params = sum(p.numel() for p in model.parameters())
bytes_per_param = next(model.parameters()).element_size()
size_mb = n_params * bytes_per_param / (1024 * 1024)
size_fp32_mb = n_params * 4 / (1024 * 1024)

print(f"Parameters:     {n_params:,}")
print(f"Precision:      {next(model.parameters()).dtype} ({bytes_per_param} bytes/param)")
print(f"Model size:     {size_mb:.0f} MB (float16)")
print(f"Full precision: {size_fp32_mb:.0f} MB (float32, estimated)")
print(f"Savings:        {size_fp32_mb / size_mb:.1f}x smaller with float16")

### Note: ONNX Runtime for faster CPU inference

For even faster CPU inference, you can use the **ONNX** (Open Neural
Network Exchange) format with the `optimum` library:

```python
# pip install optimum[onnxruntime]
from optimum.onnxruntime import ORTModelForCausalLM

model = ORTModelForCausalLM.from_pretrained(
    "HuggingFaceTB/SmolLM2-360M-Instruct",
    subfolder="onnx",
    file_name="model_q4.onnx",  # int4 quantized, ~180 MB
)
```

ONNX Runtime can be significantly faster than PyTorch for CPU
inference, and the int4 quantized model is only ~180 MB.
The `model.generate()` API is identical to standard transformers.

## 3. Text Generation Basics

Language models generate text **one token at a time**. Given a
sequence of tokens, the model predicts a probability distribution
over the next token, picks one, appends it, and repeats.

The `model.generate()` method handles this autoregressive loop.
By default, it uses **greedy decoding** — always picking the
most likely next token.

In [ ]:
prompt = "The capital of France is"
inputs = tokenizer(prompt, return_tensors="pt")

print(f"Input token IDs: {inputs['input_ids'].tolist()}")
print(f"Number of input tokens: {inputs['input_ids'].shape[1]}")

In [ ]:
output = model.generate(**inputs, max_new_tokens=50)

generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
print(f"Prompt:    {prompt!r}")
print(f"Generated: {generated_text!r}")

### Measuring generation speed

Let's measure how fast the model generates tokens. This is
typically reported in **tokens per second**.

In [ ]:
import time

prompt = "Explain in simple terms what machine learning is."
inputs = tokenizer(prompt, return_tensors="pt")
n_input_tokens = inputs["input_ids"].shape[1]

start = time.time()
output = model.generate(**inputs, max_new_tokens=100)
elapsed = time.time() - start

n_generated = output.shape[1] - n_input_tokens
tokens_per_sec = n_generated / elapsed

print(f"Generated {n_generated} tokens in {elapsed:.1f}s")
print(f"Speed: {tokens_per_sec:.1f} tokens/sec")
print(f"\n{tokenizer.decode(output[0], skip_special_tokens=True)}")

## 4. Sampling and Generation Parameters

Greedy decoding always picks the most likely token, which can
lead to repetitive or boring text. **Sampling** introduces
randomness by drawing tokens from the probability distribution.

Key parameters that control generation:

- **`temperature`**: scales the logits before softmax. Lower values
  (e.g., 0.1) make the distribution sharper (more deterministic),
  higher values (e.g., 1.5) make it flatter (more random).
- **`top_k`**: only sample from the top-k most likely tokens.
- **`top_p`** (nucleus sampling): only sample from the smallest set
  of tokens whose cumulative probability exceeds p.
- **`repetition_penalty`**: penalizes tokens that have already
  appeared, reducing repetition loops.

### Effect of temperature

In [ ]:
prompt = "Once upon a time in a land far away"
inputs = tokenizer(prompt, return_tensors="pt")

for temp in [0.1, 0.7, 1.5]:
    output = model.generate(
        **inputs,
        max_new_tokens=60,
        do_sample=True,
        temperature=temp,
    )
    text = tokenizer.decode(output[0], skip_special_tokens=True)
    print(f"\n--- Temperature {temp} ---")
    print(text)

Notice how low temperature produces more predictable, focused
text while high temperature produces more varied (and sometimes
nonsensical) text.

### top_k and top_p (nucleus sampling)

In [ ]:
prompt = "The best programming language for beginners is"
inputs = tokenizer(prompt, return_tensors="pt")

# top_k: only consider the top 10 most likely tokens at each step
output = model.generate(
    **inputs, max_new_tokens=60, do_sample=True, top_k=10, temperature=0.7
)
print("--- top_k=10 ---")
print(tokenizer.decode(output[0], skip_special_tokens=True))

# top_p: consider tokens until cumulative probability reaches 0.9
output = model.generate(
    **inputs, max_new_tokens=60, do_sample=True, top_p=0.9, temperature=0.7
)
print("\n--- top_p=0.9 ---")
print(tokenizer.decode(output[0], skip_special_tokens=True))

### Repetition penalty

Without repetition penalty, small models can get stuck in loops.
The `repetition_penalty` parameter (> 1.0) reduces the likelihood
of tokens that have already appeared:

In [ ]:
prompt = "The meaning of life is"
inputs = tokenizer(prompt, return_tensors="pt")

# Without repetition penalty (greedy decoding can loop)
output = model.generate(**inputs, max_new_tokens=80)
print("--- No repetition penalty (greedy) ---")
print(tokenizer.decode(output[0], skip_special_tokens=True))

# With repetition penalty
output = model.generate(**inputs, max_new_tokens=80, repetition_penalty=1.3)
print("\n--- repetition_penalty=1.3 (greedy) ---")
print(tokenizer.decode(output[0], skip_special_tokens=True))

### Visual comparison: 5 completions at different temperatures

In [ ]:
prompt = "In the year 2050, robots will"
inputs = tokenizer(prompt, return_tensors="pt")

temperatures = [0.1, 0.4, 0.7, 1.0, 1.5]
completions = []

for temp in temperatures:
    output = model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=True,
        temperature=temp,
        top_p=0.95,
    )
    text = tokenizer.decode(
        output[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    )
    completions.append(text)

for temp, text in zip(temperatures, completions):
    print(f"\nT={temp}: {text.strip()[:120]}")

## 5. Chat Templates

Instruction-tuned models expect input in a **structured chat
format**, not raw text. Each model family has its own template
that wraps messages with special tokens so the model knows which
parts are from the system, user, and assistant.

The `tokenizer.apply_chat_template()` method handles this
formatting automatically.

In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is the Pythagorean theorem?"},
]

# Show the raw formatted string (before tokenization)
formatted = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
print("Raw chat template:")
print(formatted)

Notice the special tokens that delimit each role. The
`add_generation_prompt=True` parameter adds the beginning of the
assistant's turn so the model knows it should start generating
a response.

In [ ]:
# Tokenize and generate
inputs = tokenizer.apply_chat_template(
    messages, return_tensors="pt", return_dict=True, add_generation_prompt=True
)
input_ids = inputs["input_ids"]
print(f"Input shape: {input_ids.shape}")

output = model.generate(**inputs, max_new_tokens=150, do_sample=False)

# Extract only the assistant's reply (skip the input tokens)
response = tokenizer.decode(
    output[0][input_ids.shape[1]:], skip_special_tokens=True
)
print(f"\nAssistant: {response}")

### Multi-turn conversation

In [ ]:
messages = [
    {"role": "system", "content": "You are a concise assistant. Keep answers short."},
    {"role": "user", "content": "What is Python?"},
    {"role": "assistant", "content": "Python is a high-level programming language known for its readability and versatility."},
    {"role": "user", "content": "What is it mainly used for?"},
]

inputs = tokenizer.apply_chat_template(
    messages, return_tensors="pt", return_dict=True, add_generation_prompt=True
)
input_ids = inputs["input_ids"]
output = model.generate(**inputs, max_new_tokens=100, do_sample=False)
response = tokenizer.decode(
    output[0][input_ids.shape[1]:], skip_special_tokens=True
)
print(f"Assistant: {response}")

## 6. Exercise: Generation with Different Parameters

**Task:** Given a prompt, generate text with 3 different temperature
settings (0.1, 0.7, 1.5) and compare the outputs. Measure the time
taken for each generation.

Use `tokenizer.apply_chat_template()` to format the prompt as a
chat message, then generate 100 tokens with `top_p=0.95` at each
temperature. Print the output and elapsed time for each.

In [ ]:
# TODO: implement the exercise





In [ ]:
# %load solutions/generate_with_params.py

## 7. The `pipeline` API (shortcut)

The `pipeline` API provides a high-level abstraction that bundles
tokenization, model inference, and decoding into a single call.
This is the easiest way to use a model for common tasks.

In [ ]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
)

In [ ]:
# Simple text completion
result = generator(
    "The three laws of robotics are",
    max_new_tokens=100,
    do_sample=False,
)
print(result[0]["generated_text"])

In [ ]:
# Chat-style usage with the pipeline
messages = [
    {"role": "user", "content": "Write a haiku about programming."},
]
result = generator(
    messages,
    max_new_tokens=60,
    do_sample=True,
    temperature=0.7,
)
print(result[0]["generated_text"][-1]["content"])

The `pipeline` API is convenient for quick experiments, but
the manual approach (tokenize, generate, decode) gives you
more control over the process.

## 8. Exercise: Text Classification via Prompting

**Task:** Use the chat model to classify BBC news snippets into
categories (business, entertainment, politics, sport, tech) using
**zero-shot classification by prompting**.

Below is a small test set of news-style snippets with ground-truth
labels. Write code to:
1. For each snippet, prompt the model to classify it
2. Extract the predicted category from the model's response
3. Compute accuracy and compare with a random baseline (1/5 = 20%)

In [ ]:
test_samples = [
    {"text": "Shares in the tech giant surged 8% after the company reported record quarterly earnings, beating analyst expectations.", "label": "business"},
    {"text": "The central bank raised interest rates by 0.25% in an effort to curb inflation, which remains above the 2% target.", "label": "business"},
    {"text": "Oil prices fell sharply as OPEC members failed to reach an agreement on production cuts at their latest meeting.", "label": "business"},
    {"text": "The new superhero blockbuster broke box office records, earning $200 million in its opening weekend worldwide.", "label": "entertainment"},
    {"text": "The singer announced a world tour spanning 50 cities after the release of her critically acclaimed new album.", "label": "entertainment"},
    {"text": "The streaming platform confirmed it will produce a new fantasy series based on the bestselling novel trilogy.", "label": "entertainment"},
    {"text": "The prime minister faced tough questions in parliament over the government's handling of the healthcare crisis.", "label": "politics"},
    {"text": "Voters head to the polls next week in what analysts predict will be the closest election in a generation.", "label": "politics"},
    {"text": "The foreign secretary met with diplomats to discuss new trade agreements following the country's exit from the bloc.", "label": "politics"},
    {"text": "The tennis champion won her fifth Grand Slam title after a thrilling three-set final that lasted over two hours.", "label": "sport"},
    {"text": "The football club confirmed the signing of the Brazilian striker for a reported fee of 60 million euros.", "label": "sport"},
    {"text": "The national team secured their place in the World Cup finals after a dramatic last-minute goal against the hosts.", "label": "sport"},
    {"text": "The company unveiled its latest smartphone featuring an AI-powered camera and a foldable display at the tech conference.", "label": "tech"},
    {"text": "Researchers developed a new chip that can perform machine learning tasks 100 times faster than existing processors.", "label": "tech"},
    {"text": "The social media platform introduced new privacy features allowing users to control who can see their online activity.", "label": "tech"},
]

In [ ]:
# TODO: implement the exercise
# For each sample in test_samples:
#   1. Create a chat message with a system prompt asking for classification
#   2. Generate the model's response
#   3. Extract and compare the predicted category with the true label
# Finally, compute and print the accuracy.





In [ ]:
# %load solutions/chat_classification.py

## 9. Quantization Formats

**Quantization** reduces model size and speeds up inference by
using lower-precision numbers for the weights:

| Format | Bits per weight | Typical size (360M params) | Notes |
|--------|:-:|:-:|---|
| fp32   | 32 | ~1.4 GB | Full precision, baseline |
| fp16   | 16 | ~720 MB | Half precision, minimal quality loss |
| int8   | 8  | ~360 MB | Good balance of size and quality |
| int4   | 4  | ~180 MB | Aggressive compression, some quality loss |

The trade-off: smaller = faster and less memory, but potentially
lower output quality. For small models like SmolLM2-360M, int4
quantization works surprisingly well.

### Quantization in practice

Our model is loaded in **float16**, which is already 2x smaller than
float32. For further compression, several quantization methods exist.
For example, this model has ONNX variants at different precisions:

In [ ]:
from huggingface_hub import list_repo_tree

onnx_files = [
    f for f in list_repo_tree(model_name, path_in_repo="onnx")
    if f.rfilename.endswith(".onnx")
]
print("Available ONNX quantization variants:")
for f in onnx_files:
    size_mb = f.size / (1024 * 1024)
    print(f"  {f.rfilename:40s}  {size_mb:7.1f} MB")

print(f"\nOur model (float16 PyTorch):  ~{sum(p.numel() for p in model.parameters()) * 2 / 1024 / 1024:.0f} MB")

### Measuring our model's generation speed

Let's benchmark our float16 model:

In [ ]:
import time

prompt = "Explain what a neural network is in simple terms."
messages = [{"role": "user", "content": prompt}]
inputs = tokenizer.apply_chat_template(
    messages, return_tensors="pt", return_dict=True, add_generation_prompt=True
)
input_ids = inputs["input_ids"]
n_input = input_ids.shape[1]

start = time.time()
with torch.no_grad():
    output = model.generate(**inputs, max_new_tokens=100, do_sample=False)
elapsed = time.time() - start
n_generated = output.shape[1] - n_input

print(f"Generated {n_generated} tokens in {elapsed:.1f}s ({n_generated/elapsed:.1f} tok/s)")
print(f"\n{tokenizer.decode(output[0][n_input:], skip_special_tokens=True)}")

### Quantization ecosystems

Popular quantization formats and tools for running LLMs efficiently:

- **ONNX Runtime** (`optimum[onnxruntime]`): portable format,
  supports int4/int8 quantization, fast CPU inference (see note in Section 2)
- **GGUF** (llama.cpp / Ollama): very popular for local inference,
  supports CPU and GPU, easy to use via Ollama CLI
- **bitsandbytes**: GPU-focused quantization (int8, int4) integrated
  directly into HuggingFace transformers
- **GPTQ**: post-training quantization designed for GPU inference
- **AWQ**: activation-aware weight quantization, often better quality
  than GPTQ at the same bit width

## 10. Going Further

### Courses and tutorials
- [HuggingFace smol-course](https://github.com/huggingface/smol-course) —
  hands-on course covering fine-tuning, alignment, and deployment of small LLMs
- [mlabonne/llm-course](https://github.com/mlabonne/llm-course) —
  comprehensive LLM course with notebooks

### Easier local inference
- [Ollama](https://ollama.ai) — one-command tool to run LLMs locally
  (uses GGUF format under the hood)

### Larger models to try
- **Qwen2.5-1.5B-Instruct** — 1.5B parameters, good multilingual support
- **Phi-3.5-mini** — 3.8B parameters from Microsoft, strong reasoning
- **Llama 3.2-1B** — 1B parameter model from Meta

### Advanced ONNX inference
- [ONNX Runtime GenAI](https://github.com/microsoft/onnxruntime-genai) —
  optimized generation loop for ONNX models with features like
  continuous batching and KV-cache management